# Avito DS Bootcamp: поиск и ранжирование объявлений

Цель — для каждого запроса вернуть 50 объявлений и максимизировать Recall@50. Решение v4 сначала собирает кандидатов несколькими способами, затем ранжирует их CatBoost YetiRank.

Notebook — исполняемый walkthrough того же кода, который запускается командой `python scripts/reproduce_v4.py --data-dir data`.

## 1. Настройки воспроизводимости

Все пути относительны корню проекта. Seed, модель эмбеддингов и параметры YetiRank зафиксированы в config.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import PipelineConfig
from src.data import load_data, build_validation_and_corpus, build_ranker_train, build_internal
from src.features import FINAL_FEATURE_NAMES, FIELD_FEATURE_NAMES, GEO_FEATURE_NAMES
from src.geo import build_transition_table
from src.pipeline import check_environment, reproduce
from src.retrieval import sparse_topk
from src.utils import normalize_text

ROOT = Path.cwd()
DATA = ROOT / "data"
CONFIG = PipelineConfig.from_root(ROOT, DATA, ROOT / "answer_v4.csv")
SEED = CONFIG.seed
check_environment(CONFIG)

## 2. Загрузка данных

`train` содержит наблюдавшиеся пары запрос–выбранное объявление. В benchmark запросы и корпус объявлений разделены.

In [ ]:
train_df, benchmark_queries_df, benchmark_items_df = load_data(DATA)
print("train:", train_df.shape)
print("benchmark queries:", benchmark_queries_df.shape)
print("benchmark items:", benchmark_items_df.shape)

## 3. Короткий EDA

Смотрим частотность запросов, заполненность фильтров и географию корпуса. Эти графики не участвуют в модели.

In [ ]:
query_frequency = normalize_text(train_df.search_query).value_counts()
print(query_frequency.describe(percentiles=[.5, .9, .95, .99]))
print("Фильтры train:", normalize_text(train_df.search_infm_params_text).ne("").mean())
print("Фильтры benchmark:", normalize_text(benchmark_queries_df.search_infm_params_text).ne("").mean())

In [ ]:
geo = benchmark_items_df[["item_longitude", "item_latitude"]].apply(pd.to_numeric, errors="coerce").dropna()
plt.figure(figsize=(8, 5))
plt.hexbin(geo.item_longitude, geo.item_latitude, gridsize=100, bins="log", mincnt=1)
plt.xlabel("долгота"); plt.ylabel("широта"); plt.title("Расположение benchmark-объявлений")
plt.colorbar(label="log числа объявлений"); plt.show()

## 4. Validation без объединения независимых событий

В train нет `query_id`. Поэтому один пример — уникальная пара `(видимая сигнатура поиска, item_id)` с одним известным положительным объектом. Локальная метрика называется **single-positive offline Recall@50 proxy**.

Development, internal и корпус строятся детерминированно. Закрытый holdout после ранней проверки v1 для настройки больше не используется.

In [ ]:
split_dir = CONFIG.cache_dir / "v4" / "splits"
if (split_dir / "development.pkl").exists():
    development = pd.read_pickle(split_dir / "development.pkl")
    corpus = pd.read_pickle(split_dir / "corpus.pkl")
    ranker_train = pd.read_pickle(split_dir / "ranker_train.pkl")
else:
    development, _unused_holdout, corpus = build_validation_and_corpus(train_df, SEED)
    ranker_train = build_ranker_train(train_df, development, corpus, seed=SEED)
internal = build_internal(train_df, development, ranker_train, corpus, seed=SEED)
print({"train": len(ranker_train), "internal": len(internal), "development": len(development), "corpus": len(corpus)})

## 5. Минимальный baseline: word TF-IDF по заголовку

Это первый понятный ориентир. Он ищет объявления по косинусному сходству запроса и заголовка, без нейросетей и метаданных.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, dtype=np.float32)
item_matrix = vectorizer.fit_transform(normalize_text(corpus.item_title_raw))
query_matrix = vectorizer.transform(development.search_query.fillna(""))
baseline_positions, baseline_scores = sparse_topk(query_matrix, item_matrix, top_k=50)
item_index = pd.Index(corpus.item_id.astype(str))
positive = item_index.get_indexer(development.item_id.astype(str))
baseline_hit50 = (baseline_positions == positive[:, None]).any(axis=1).mean()
print("TF-IDF single-positive Hit@50:", round(float(baseline_hit50), 4))

## 6. Основные источники кандидатов

Final v4 объединяет TF-IDF, Giga Embeddings, локальные ветки, поиск в радиусе 50 км и поведенческую географию. Каждый источник сохраняет позиции и оценки; повторный запуск читает их из `cache/`.

In [ ]:
from src.config import ALL_SOURCES
pd.Series(ALL_SOURCES, name="candidate source").to_frame()

## 7. Local и geo retrieval

`Giga_local` ищет только в точной локации, `Giga_geo50` — около географического центра локации. Это мягкие источники кандидатов, а не жёсткие фильтры.

In [ ]:
from src.retrieval import restricted_dense, geo50_dense
print(restricted_dense.__doc__ or "Точный dense-поиск внутри группы location_id")
print(geo50_dense.__doc__ or "Dense-поиск среди объявлений в радиусе 50 км")

## 8. Behavioral geo redirect

По истории оцениваем, в каких локациях пользователи из каждой поисковой локации реально выбирали услуги. Это позволяет находить межгородские положительные примеры.

In [ ]:
local_items = train_df[[c for c in train_df if c == "item_id" or c.startswith("item_")]].drop_duplicates("item_id")
transitions, transition_summary = build_transition_table(train_df, local_items)
transition_summary[["observations", "top1_probability", "destinations", "entropy"]].describe()

## 9. Объединение кандидатов

Кандидаты всех источников объединяются по позиции объявления. Обычно это около 800 объектов на benchmark-запрос; final-модель должна выбрать 50.

In [ ]:
def union_positions(source_rows):
    return np.unique(np.concatenate([row[row >= 0] for row in source_rows]))

## 10. Query-relative и field-level признаки

131 базовый признак описывает ранги, оценки и согласие источников внутри запроса. Ещё 11 признаков относятся к behavioral geo redirect, а 43 — к совпадениям запроса с title, params и description.

In [ ]:
print("Всего признаков:", len(FINAL_FEATURE_NAMES))
print("Behavioral geo:", len(GEO_FEATURE_NAMES))
print("Field interactions:", len(FIELD_FEATURE_NAMES))
pd.Series(FIELD_FEATURE_NAMES[:15], name="пример field-level признаков").to_frame()

## 11. YetiRank

Обучающая группа соответствует одному запросу. В каждой группе есть выбранное объявление и до 100 отрицательных кандидатов, взятых из того же распределения источников, что и на inference.

In [ ]:
CONFIG.catboost_params

## 12. Internal и development

При запуске с `--retrain` pipeline заново строит training matrix, обучает YetiRank и может оценить его на зафиксированных разбиениях. Ожидаемые контрольные значения E4: internal 0.8878, development 0.8940, development coverage 0.9642.

In [ ]:
# Полная проверка действительно пересчитывает предсказания internal/dev.
# Она дорогая, поэтому включается осознанно вместе с переобучением.
RUN_LOCAL_EVALUATION = False
if RUN_LOCAL_EVALUATION:
    local_report = reproduce(CONFIG, retrain=True)
    display(local_report["local_metrics"])
else:
    print("Для полного пересчёта: RUN_LOCAL_EVALUATION = True")

## 13. Benchmark inference и создание submission

Следующий вызов является реальным pipeline. С готовым кэшем он использует сохранённые дорогие стадии; без кэша строит отсутствующие retrieval-ветки. Результат проходит строгую проверку 2452 × 50.

In [ ]:
run_report = reproduce(CONFIG, retrain=False)
run_report

## 14. Официальный результат

Полученный `answer_v4.csv` дал официальный Recall@50 **0.844391**. Benchmark не использовался для выбора признаков или параметров.